# Entrenar YOLOv8 para deteccion de patentes chilenas
---
1. Subir el dataset ZIP a Drive
2. Conectar runtime GPU (Runtime → Change runtime type → T4 GPU)
3. Ejecutar todo (Ctrl+F9)

In [ ]:
!pip install ultralytics pyyaml -q

In [ ]:
import os, zipfile, shutil, yaml
from pathlib import Path
from ultralytics import YOLO

# ─── CONFIG ───────────────────────────────────
MODEL_SIZE = "yolov8n.pt"   # nano (rapido) | yolov8s.pt (preciso)
IMGSZ = 640
EPOCHS = 150
BATCH = 16
PATIENCE = 25
# ───────────────────────────────────────────────

# Para CONTINUAR entrenando desde best.pt anterior, subilo aca:
# from google.colab import files
# uploaded = files.upload()
# PREVIOUS_PT = list(uploaded.keys())[0]
PREVIOUS_PT = None  # None = desde cero

DATASET_DIR = Path("/content/dataset")
shutil.rmtree(DATASET_DIR, ignore_errors=True)

# Subir uno o mas ZIPs de datasets (seleccionar varios con Ctrl+Click)
from google.colab import files
uploaded = files.upload()

# Extraer cada ZIP a una subcarpeta
for fname in uploaded.keys():
    src_name = Path(fname).stem
    print(f"Extrayendo {fname}...")
    with zipfile.ZipFile(fname) as z:
        z.extractall(str(DATASET_DIR / src_name))

# Unificar splits de todas las fuentes
for split in ("train", "valid", "test"):
    dest_img = DATASET_DIR / split / "images"
    dest_lbl = DATASET_DIR / split / "labels"
    dest_img.mkdir(parents=True, exist_ok=True)
    dest_lbl.mkdir(parents=True, exist_ok=True)
    for src in DATASET_DIR.iterdir():
        if src == DATASET_DIR or not src.is_dir(): continue
        s_img = src / split / "images"
        s_lbl = src / split / "labels"
        if s_img.exists():
            for f in s_img.iterdir():
                if f.suffix.lower() in (".jpg",".jpeg",".png"):
                    f.rename(dest_img / f"{src.name}_{f.name}")
        if s_lbl.exists():
            for f in s_lbl.iterdir():
                f.rename(dest_lbl / f"{src.name}_{f.name}")
    total_img = len(list(dest_img.glob("*.*")))
    print(f"  {split}: {total_img} imagenes")

# Crear data.yaml unificado
data_yaml = DATASET_DIR / "data.yaml"
data_yaml.write_text(yaml.dump({
    "train": str((DATASET_DIR / "train" / "images").resolve()),
    "val": str((DATASET_DIR / "valid" / "images").resolve()),
    "nc": 1,
    "names": ["plate"]
}))
print(f"data.yaml creado en {data_yaml}")

In [ ]:
model = YOLO(PREVIOUS_PT if PREVIOUS_PT else MODEL_SIZE)
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device="cuda",
    workers=4,
    amp=True,
    name="plate_detector",
    exist_ok=True,
)

In [ ]:
best_pt = Path("runs/detect/plate_detector/weights/best.pt")
model = YOLO(str(best_pt))
model.export(format="onnx", imgsz=IMGSZ, half=True)
onnx_path = best_pt.parent / "best.onnx"
print(f"ONNX: {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
from google.colab import files
print("Descargando best.onnx...")
files.download(str(onnx_path))
print("Descargando best.pt (para re-entrenar despues)...")
files.download(str(best_pt))
print("ONNX -> assets/models/yolov8_plate.onnx")
print("PT   -> training/ para futuros re-entrenamientos")